# Core Data Types and Type Conversion

**Topic:** `1.5` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Everything that enters a robot arrives as text. This topic covers the core built-in types, the traps of truthiness and floating point, and the conversion ladder that makes external data safe to use.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

## 1. Why This Topic Matters

Type conversion is the seam where a robot meets the outside world: serial lines, HTTP parameters, configuration files and operator input all produce text. Get this boundary right and the rest of a system can trust its inputs; get it wrong and a single corrupt field silently becomes a wrong motor command.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Name the core built-in types and say which are mutable.
2. Convert external text safely with a conversion ladder.
3. Explain why bool is a subclass of int and what it breaks.
4. Avoid truthiness bugs on numeric values.
5. Compare floats with a tolerance and know when Decimal is worth it.
6. Parse a telemetry frame without losing the valid channels.

## 3. Prerequisites

This topic assumes you already have:

- Topic 1.4 Variables, Naming Conventions, and Dynamic Typing

## 4. Mental Model

**Mental model: a typed slot, but the slot can be re-labelled.** Built-in values are sealed - an `int` or a `str` cannot be modified in place, so handing one to a function is safe. But the *name* holding the value is not sealed: you can point it at a different type at any moment. Safety comes from converting deliberately at the edges, not from the language preventing a mistake.

| Value | Falsy? | Mutable? | Safe to share? |
| --- | --- | --- | --- |
| `48` | no | no | yes |
| `0` | yes | no | yes, but beware `if value:` |
| `'48.5'` | no | no | yes |
| `''` | yes | no | yes |
| `None` | yes | n/a | yes - the standard 'no value' marker |
| `[1]` | no | yes | only if copied or not mutated |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Type | The class of a value, reported by type(). |
| Built-in type | A type implemented by the interpreter, such as int or str. |
| Conversion | Explicitly producing a new value of another type. |
| Mutability | Whether a value's contents can be changed in place. |
| Truthiness | The implicit boolean value Python assigns to any object. |
| Falsy | One of the few values that evaluate as False. |
| None | The singleton marker for 'no value'. |
| Arithmetic promotion | Mixing int and float promotes the int to float. |
| IEEE 754 double | The binary floating-point format Python floats use. |
| Decimal | A standard-library decimal type for exact base-10 arithmetic. |

## 6. Conceptual Explanation

Python ships a small set of built-in types that cover almost everything a robot needs. `int` and `float` for measurements, `str` for text arriving from sensors and operators, `bool` for decisions, and `None` for *absent*. `list`, `tuple`, `dict` and `set` are covered in Module 3. Every value is an object with a type, and `type()` will always tell you which one you are holding - which makes introspection a practical debugging tool rather than a curiosity.

In [ ]:
samples = [42, 3.14, 'rover-01', True, False, None, (1, 2), {'a': 1}, {1, 2}]
for value in samples:
    print(f'{repr(value):<12} {type(value).__name__}')

Two properties matter more than the rest at this stage. First, **immutability**: `int`, `float`, `str` and `bool` cannot be modified in place, so a function receiving one can never damage it. Second, **truthiness**: every value has a truth value, and only a small set of them are falsy. That is why `if value:` is almost always a bug when the value is a number and zero is legitimate.

In [ ]:
falsy = [0, 0.0, '', [], {}, set(), None, False]
truthy = [1, -1, 0.5, '0', [0], {'k': 0}, True]

print('falsy  :', [bool(v) for v in falsy])
print('truthy :', [bool(v) for v in truthy])

distance_m = 0.0
if distance_m:               # a robot that has not moved yet
    print('moving')
else:
    print('stationary (distance is exactly zero)')

> **bool is a subclass of int**
>
> Because True is 1 and False is 0 under the hood, isinstance(True, int) is True. That bit of inheritance is a frequent source of validation bugs: a check that accepts any number will also accept a boolean unless you exclude it explicitly.

## 7. Formal Theory

Python implements a **numeric tower**: `bool` is a subclass of `int`, and `int` is conceptually a parent of `float`. Mixing an `int` and a `float` in an arithmetic operation promotes the `int` to a `float` rather than raising. This is why `1 + 2.0` is `3.0` and not `3`, and why a division always produces a float.

```text
int + float  ->  float        (the int is promoted, never truncated)
```

Integers are arbitrary precision: Python will represent an integer of any length, so overflow is not a failure mode for `int`. Floats are IEEE 754 double-precision, which means they use binary floating point and therefore cannot represent most decimal fractions exactly. That is the origin of results that surprise beginners.

In [ ]:
print('0.1 + 0.2 =', 0.1 + 0.2)
print('equal to 0.3?', 0.1 + 0.2 == 0.3)
print('close enough?', abs((0.1 + 0.2) - 0.3) < 1e-9)

from decimal import Decimal
print('Decimal     :', Decimal('0.1') + Decimal('0.2'))

huge = 2 ** 100
print('2 ** 100    :', huge)
print('as float    :', float(huge))

| Type | Mutable | Example | Typical robotics use |
| --- | --- | --- | --- |
| `int` | no | 48 | counts, identifiers, whole Wh |
| `float` | no | 48.5 | sensor readings, power |
| `str` | no | '48.5' | raw sensor and operator text |
| `bool` | no | True | flags and decisions |
| `NoneType` | n/a | None | 'no value yet' |
| `list` | yes | [1, 2] | sequences of readings |
| `dict` | yes | {'a': 1} | keyed channel data |

## 8. Syntax

Conversion is always explicit. There is no implicit cast in Python.

In [ ]:
battery_pct = '87.5'      # text from a sensor bus

print('as text :', battery_pct)
print('to float:', float(battery_pct))
print('to int  :', int(float(battery_pct)))   # int() alone would raise
print('to str  :', str(48.0))
print('to bool :', bool(battery_pct))        # a non-empty string is True

try:
    int('87.5')
except ValueError as exc:
    print('int on a decimal string raises:', exc)

The anti-pattern:

```text
pct = '87.5'
pct = pct * 2          # string repetition! '87.587.5', not 175.0

pct = float(pct) * 2   # convert first, then compute
```

> **int() does not round**
>
> int(87.5) raises ValueError rather than returning 87. It also truncates silently for floats: int(-1.9) is -1, not -2. Use round() when you mean nearest.

## 9. Basic Examples

**Example 1 - the four scalar types in use.**

In [ ]:
count = 3                    # int
voltage = 12.6               # float
label = 'front-left'         # str
active = True                # bool
missing = None               # NoneType

for name, value in (
    ('count', count), ('voltage', voltage), ('label', label),
    ('active', active), ('missing', missing),
):
    print(f'{name:<8} {type(value).__name__:<9} {value!r}')

**Example 2 - useful string methods for parsing.**

In [ ]:
raw = '  12.60 m '
print('stripped :', repr(raw.strip()))
print('numeric  :', raw.strip().rstrip('m').strip())
print('parsed   :', float(raw.strip().rstrip('m').strip()))
print('split    :', 'a,b,c'.split(','))
print('joined   :', '-'.join(['a', 'b', 'c']))

## 10. Intermediate Examples

The reliable pattern for external data is a **conversion ladder**: try the narrowest type first, widen on failure, and finish with the string itself. This is what turns a configuration value into something your code can trust.

In [ ]:
def coerce(token: str):
    try:
        return int(token)
    except ValueError:
        pass
    try:
        return float(token)
    except ValueError:
        return token


for token in ('48', '48.5', 'auto', ''):
    value = coerce(token)
    print(f'{token!r:<8} -> {type(value).__name__:<5} {value!r}')

> **Order matters**
>
> Trying int first means '48' becomes an int rather than 48.0. That is usually what a configuration reader wants, because a whole number of Wh should not gain a spurious decimal point.

## 11. Advanced Examples

Decimals that look exact are not exact in binary floating point, so equality checks against a literal can fail. Two correct responses exist: compare with a tolerance, or use `decimal.Decimal` when the decimal representation is the point.

In [ ]:
from decimal import Decimal

left, right = 0.1, 0.2
print('float sum      :', repr(left + right))
print('float == 0.3   :', left + right == 0.3)

print('within 1e-9    :', abs((left + right) - 0.3) < 1e-9)

exact = Decimal('0.1') + Decimal('0.2')
print('decimal sum    :', exact)
print('decimal == 0.3 :', exact == Decimal('0.3'))

Use `Decimal` for money-like accounting and for measurements where the decimal value is contractual. Use floats for ordinary sensor readings, and compare them with a tolerance rather than with `==`.

## 12. Code Walkthrough

Below is a parsing function for raw telemetry text - the exact shape a robot receives over a serial line or an HTTP query string.

In [ ]:
"""readings.py - turn raw telemetry text into validated floats."""
from __future__ import annotations

MISSING = None


def parse_readings(text: str) -> tuple[dict, list[str]]:
    """Parse 'channel=value' pairs into floats.

    Returns (readings, problems). A malformed pair is reported by
    name and excluded, so one bad sensor never loses the good ones.
    """
    readings: dict = {}
    problems: list[str] = []
    for chunk in text.split(';'):
        chunk = chunk.strip()
        if not chunk:
            continue
        name, separator, raw = chunk.partition('=')
        if not separator:
            problems.append(f'{chunk!r}: missing =')
            continue
        name = name.strip()
        try:
            readings[name] = float(raw.strip())
        except ValueError:
            problems.append(f'{name}: {raw.strip()!r} is not a number')
    return readings, problems

The key decision is to **not raise**. A telemetry stream will eventually contain a corrupt field, and a parser that dies on the first bad value takes down every other channel with it. Recording the problem and continuing keeps the rest of the data usable.

## 13. Line-by-Line Explanation

1. The docstring states the partial-success contract up front: the caller is told it gets usable data plus a list of complaints.
2. `text.split(';')` produces the chunks; blank chunks are skipped so a trailing separator is harmless.
3. `partition('=')` splits on the first equals sign only, so a value containing '=' is not torn apart.
4. A chunk without '=' is reported rather than parsed, because there is no name to attach the value to.
5. float() is called inside try/except ValueError only, so a genuinely wrong type elsewhere in the program still surfaces as an error.
6. Only successfully converted channels enter `readings`, so downstream code can trust every value it contains.

## 14. Common Mistakes

**Mistake 1 - multiplying a numeric string.**

```text
pct = '87.5'
pct * 2            # '87.587.5' - string repetition

float(pct) * 2     # 175.0
```

**Mistake 2 - int() on a decimal string.**

```text
int('87.5')        # ValueError: invalid literal for int()

int(float('87.5')) # 87 - converts first, then truncates
```

**Mistake 3 - trusting a value because a bool check passed.**

```text
if value:                  # 0.0 and '' both skip the branch
    publish(value)

if value is not None:      # explicit absence check
    publish(value)
```

**Mistake 4 - comparing floats with ==.**

```text
0.1 + 0.2 == 0.3          # False

abs(0.1 + 0.2 - 0.3) < 1e-9   # True - compare with a tolerance
```

## 15. Debugging Techniques

When a conversion misbehaves, the useful questions are: what type is it, what does it look like when printed, and what does the conversion actually do?

In [ ]:
samples = ['87.5', '  87.5  ', '87,5', 'n/a', '', None, True]
for value in samples:
    try:
        converted = float(value)
        print(f'{value!r:<10} -> {converted!r}')
    except (TypeError, ValueError) as exc:
        print(f'{value!r:<10} -> {type(exc).__name__}: {exc}')

Note the two exception types. `ValueError` means the text is not a number, while `TypeError` means the object has no sensible numeric interpretation at all. Catching both is right at a system boundary and wrong in the middle of your own logic, where a TypeError is a bug you want to see.

> **Locate the boundary**
>
> When a value has the wrong type, the bug is almost never where the error was raised. It is where the value entered the program. Trace backwards to the first place it was created, not the last place it was used.

## 16. Best Practices

- Convert external text exactly once, at the boundary where it enters.
- Use a conversion ladder (int, then float, then str) for configuration.
- Never rely on truthiness for numeric values; compare explicitly.
- Check `value is not None` when None is a legitimate value.
- Compare floats with a tolerance, never with ==.
- Remember bool is a subclass of int when validating numbers.
- Report conversion failures with the offending name and text.
- Keep one bad value from destroying the whole batch.

## 17. Performance Considerations

Type conversion is not free, but it is cheap relative to I/O: parsing a few thousand sensor readings is dominated by the file or socket read that produced them. Optimising the `float()` call itself is almost never the right investment. The optimisation that *does* pay is avoiding repeated conversion of the same value - if a channel is read in a thousand iterations, convert once outside the loop and reuse the result.

`Decimal` is an order of magnitude slower than `float` because it performs arbitrary-precision decimal arithmetic. Use it where correctness demands it - accounting, contractual tolerances - and measure before applying it to a high-rate telemetry path, where a float with an explicit tolerance is both faster and more honest about the precision you actually have.

> **Batch, do not stream-convert**
>
> If you must convert a very large batch, prefer a library routine that parses the whole column at once over a Python-level loop calling float() per item.

## 18. Pythonic Approaches

Type checks: comparing against type objects versus isinstance.

In [ ]:
value = 48.0

# before: an exact type comparison
print('exact  :', type(value) == float)

# after: isinstance, which also accepts subclasses
print('isinstance:', isinstance(value, float))
print('bool is an int:', isinstance(True, int))

# excluding bool explicitly when a number is required
def numeric(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


print('numeric(48.0):', numeric(48.0))
print('numeric(True):', numeric(True))

## 19. Robotics Connection

Every value a robot receives from the world is text first: a serial line, an HTTP query parameter, a YAML file. The simulator returns real floats, which is precisely why the conversion step must be exercised separately - it is the one part of the pipeline that never gets tested by the hardware.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
state = robot.status()

# render a real reading as text, then parse it back
wire = f"battery_pct={state['battery_pct']};travelled_m={state['travelled_m']}"
print('on the wire:', wire)
readings, problems = parse_readings(wire)
print('parsed      :', readings)
print('problems    :', problems)

## 20. Engineering Example

A validating converter suitable for a system boundary: it reports what failed, applies an optional range, and never raises for bad external data.

In [ ]:
def read_number(raw, *, low=None, high=None, default=None):
    """Convert external text to a float, reporting rather than raising."""
    if raw is None:
        return default
    if isinstance(raw, bool):
        return default
    if isinstance(raw, (int, float)):
        value = float(raw)
    else:
        try:
            value = float(str(raw).strip())
        except ValueError:
            return default
    if low is not None and value < low:
        return default
    if high is not None and value > high:
        return default
    return value

## 21. Guided Practice

1. Print type() and bool() for 0, 0.0, '', '0', None and False, and explain each result.
2. Try int(), float() and bool() on the string '48.5' and record which succeed.
3. Compute 0.1 + 0.2 - 0.3 and print it in full precision before rounding.
4. Write `is_number(value)` that returns True for int and float but False for bool, then test it on five values.

In [ ]:
import math

for value in (0, 0.0, '', '0', None, False):
    print(f'{value!r:<8} bool={bool(value)}')

print('int(48.5) raises:', end=' ')
try:
    int('48.5')
except ValueError:
    print('yes')

print('residual:', math.fsum([0.1, 0.2]) - 0.3)

def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** Why is `isinstance(True, int)` True in Python?
- **Q1.** Why should a telemetry parser report a bad channel rather than raise?
- **Q1.** Why does the standard library provide decimal.Decimal?
- **Q1.** Why validate ranges after parsing when the simulator already clamps readings?

## 23. Summary

Python's core types are few and deliberately blunt: `int`, `float`, `str`, `bool` and `None`, with the containers following in Module 3. Four of these are immutable, which makes them safe to pass around freely; the type lives on the value rather than the name, so typing is dynamic and conversion is always explicit.

Two traps account for most beginner bugs here. Truthiness bites because every value has a boolean interpretation and only a small set are falsy, so `if value:` silently skips a legitimate reading of `0.0`. And floats bite because binary floating point cannot represent most decimal fractions exactly, so `0.1 + 0.2 == 0.3` is False and must be replaced by a tolerance comparison or by `Decimal`.

The professional pattern is the conversion ladder: at every system boundary, try the narrowest type, widen on failure, finish with the string, and record what failed rather than raising. A telemetry stream will eventually contain a corrupt field, and a parser that dies on it takes every other channel down with it. Parse defensively once, at the edge, and let the rest of the program trust its inputs.

## 24. Key Takeaways

- int, float, str, bool and None cover most robotics logic.
- Type lives on the value; the name can be rebound freely.
- Conversion in Python is always explicit - there are no implicit casts.
- A conversion ladder handles mixed configuration values safely.
- Only 0, 0.0, '', empty containers, None and False are falsy.
- bool is a subclass of int, so exclude it explicitly when validating numbers.
- Compare floats with a tolerance, or use Decimal where exactness is required.
- At a system boundary, report conversion failures instead of raising.

## 25. Further Exploration

- [Built-in types reference](https://docs.python.org/3/library/stdtypes.html)
- [numeric tower - the type hierarchy](https://docs.python.org/3/reference/numerictypes.html)
- [decimal - fixed-point and floating-point decimal](https://docs.python.org/3/library/decimal.html)
- [Truth value testing](https://docs.python.org/3/library/stdtypes.html#truth-value-testing)
- [PEP 484 - Type hints](https://peps.python.org/pep-0484/) - catching this class of bug statically.